# LeetCode 1268 — Search Suggestions System bằng Python

Link đề: https://leetcode.com/problems/search-suggestions-system/ · Độ khó: **Medium** · Chủ đề: Array, String, Binary Search, Trie, Sorting, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def suggestedProducts(self, products: list[str], searchWord: str) -> list[list[str]]:
        ...
```

## 1. Mục tiêu học

1. Hiểu tính chất then chốt: sau khi **sắp xếp**, các chuỗi có cùng tiền tố nằm thành **một đoạn liên tiếp**.
2. Đi từ **lọc lại toàn bộ** cho mỗi tiền tố → **lọc dần** → **tìm kiếm nhị phân** (`bisect`) → **hai con trỏ** → **Trie lưu sẵn top-3**.
3. Dùng thành thạo `str.startswith`, so sánh chuỗi theo từ điển, `bisect.bisect_left`, và cài đặt Trie bằng `dict`.
4. Biết cân nhắc chi phí xây dựng (sort, Trie) với chi phí mỗi truy vấn.
5. Tránh lỗi: quên sắp xếp, dừng sớm khi hết gợi ý (thiếu danh sách rỗng), `bisect` mà không kiểm tra tiền tố, truy cập ký tự vượt độ dài khi dùng hai con trỏ.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `products` và chuỗi `searchWord`. Thiết kế hệ thống gợi ý **tối đa 3** sản phẩm sau **mỗi ký tự** được gõ của `searchWord`. Sản phẩm gợi ý phải có **tiền tố chung** với phần đã gõ. Nếu có hơn 3 sản phẩm phù hợp, chọn 3 sản phẩm **nhỏ nhất theo thứ tự từ điển**.

Trả về danh sách các danh sách gợi ý, lần lượt sau khi gõ ký tự thứ 1, 2, …, `len(searchWord)`.

**Ví dụ 1:** `products = ["mobile","mouse","moneypot","monitor","mousepad"]`, `searchWord = "mouse"` →
`[["mobile","moneypot","monitor"],["mobile","moneypot","monitor"],["mouse","mousepad"],["mouse","mousepad"],["mouse","mousepad"]]`.

**Ví dụ 2:** `products = ["havana"]`, `searchWord = "havana"` → `[["havana"]] * 6`.

**Ràng buộc:** `1 <= len(products) <= 1000`; `1 <= len(products[i]) <= 3000`; tổng độ dài ≤ `2·10^4`; các sản phẩm **khác nhau**, chỉ gồm chữ thường; `1 <= len(searchWord) <= 1000`, chữ thường.

In [1]:
import bisect
import random
import string
import time
from typing import List

EX1 = (["mobile", "mouse", "moneypot", "monitor", "mousepad"], "mouse",
       [["mobile", "moneypot", "monitor"], ["mobile", "moneypot", "monitor"],
        ["mouse", "mousepad"], ["mouse", "mousepad"], ["mouse", "mousepad"]])
EX2 = (["havana"], "havana", [["havana"]] * 6)
EX3 = (["bags", "baggage", "banner", "box", "cloths"], "bags",
       [["baggage", "bags", "banner"], ["baggage", "bags", "banner"], ["baggage", "bags"], ["bags"]])
EX4 = (["havana"], "tatiana", [[] for _ in range(7)])
examples = [EX1, EX2, EX3, EX4]
for prods, w, want in examples:
    print(w, "->", want)

mouse -> [['mobile', 'moneypot', 'monitor'], ['mobile', 'moneypot', 'monitor'], ['mouse', 'mousepad'], ['mouse', 'mousepad'], ['mouse', 'mousepad']]
havana -> [['havana'], ['havana'], ['havana'], ['havana'], ['havana'], ['havana']]
bags -> [['baggage', 'bags', 'banner'], ['baggage', 'bags', 'banner'], ['baggage', 'bags'], ['bags']]
tatiana -> [[], [], [], [], [], [], []]


(Ví dụ 3, 4 là ví dụ bổ sung tự dựng — đã kiểm chứng bằng brute force ở mục 12; nên đối chiếu lại đề gốc nếu muốn chắc chắn chúng có trong đề.)

## 3. Kiến thức nền

### 3.1 So sánh chuỗi & sắp xếp theo từ điển

Python so sánh chuỗi từng ký tự từ trái sang; nếu một chuỗi là **tiền tố** của chuỗi kia thì chuỗi ngắn hơn nhỏ hơn: `"mouse" < "mousepad"`.

In [2]:
prods = EX1[0]
print(sorted(prods))
print('"mouse" < "mousepad":', "mouse" < "mousepad", '| "mon" < "mou":', "mon" < "mou")

['mobile', 'moneypot', 'monitor', 'mouse', 'mousepad']
"mouse" < "mousepad": True | "mon" < "mou": True


### 3.2 `str.startswith` và cắt tiền tố

`s.startswith(p)` kiểm tra tiền tố trong `O(len(p))`. Tiền tố thứ `i` của `searchWord` là `searchWord[:i]`.

In [3]:
w = "mouse"
print([w[:i] for i in range(1, len(w) + 1)])
print("mousepad".startswith("mous"), "monitor".startswith("mou"))

['m', 'mo', 'mou', 'mous', 'mouse']
True False


### 3.3 Đoạn liên tiếp sau khi sắp xếp

Trong danh sách đã sắp xếp, mọi chuỗi bắt đầu bằng `p` nằm liền nhau, và chúng đứng ngay tại (hoặc sau) vị trí mà `p` sẽ được chèn vào. Đây là nền tảng của Cách 3 và 4.

In [4]:
sp = sorted(EX1[0])
for p in ["m", "mo", "mou", "mon", "x"]:
    idx = [i for i, s in enumerate(sp) if s.startswith(p)]
    print(f"tiền tố {p!r:6}: các chỉ số khớp {idx}")

tiền tố 'm'   : các chỉ số khớp [0, 1, 2, 3, 4]
tiền tố 'mo'  : các chỉ số khớp [0, 1, 2, 3, 4]
tiền tố 'mou' : các chỉ số khớp [3, 4]
tiền tố 'mon' : các chỉ số khớp [1, 2]
tiền tố 'x'   : các chỉ số khớp []


### 3.4 `bisect.bisect_left`

`bisect_left(a, x)` trả về vị trí **đầu tiên** `i` sao cho `a[i] >= x` (danh sách `a` đã sắp xếp), trong `O(log n)` lần so sánh. Với `x = p` (tiền tố), mọi chuỗi bắt đầu bằng `p` đều `>= p`, còn mọi chuỗi `< p` chắc chắn không bắt đầu bằng `p` ⇒ đoạn khớp bắt đầu đúng tại `bisect_left(a, p)`.

In [5]:
for p in ["m", "mou", "mon", "moz", "a"]:
    i = bisect.bisect_left(sp, p)
    print(f"bisect_left(sp, {p!r:6}) = {i} -> {sp[i:i + 3]}")

bisect_left(sp, 'm'   ) = 0 -> ['mobile', 'moneypot', 'monitor']
bisect_left(sp, 'mou' ) = 3 -> ['mouse', 'mousepad']
bisect_left(sp, 'mon' ) = 1 -> ['moneypot', 'monitor', 'mouse']
bisect_left(sp, 'moz' ) = 5 -> []
bisect_left(sp, 'a'   ) = 0 -> ['mobile', 'moneypot', 'monitor']


### 3.5 Trie (cây tiền tố) bằng `dict`

Mỗi nút là một `dict` từ ký tự sang nút con. Đi theo các ký tự của tiền tố là tới nút đại diện cho tiền tố đó. Ta có thể lưu thêm ở mỗi nút danh sách "top-3" để trả lời ngay.

In [6]:
trie = {}
for word in ["mob", "mou", "mon"]:
    node = trie
    for ch in word:
        node = node.setdefault(ch, {})
    node["$"] = word            # đánh dấu kết thúc từ
print(trie)

{'m': {'o': {'b': {'$': 'mob'}, 'u': {'$': 'mou'}, 'n': {'$': 'mon'}}}}


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**. Sắp xếp: `["mobile", "moneypot", "monitor", "mouse", "mousepad"]`.

| Đã gõ | Các sản phẩm khớp (đã sắp) | 3 đầu tiên |
|---|---|---|
| `m` | cả 5 | mobile, moneypot, monitor |
| `mo` | cả 5 | mobile, moneypot, monitor |
| `mou` | mouse, mousepad | mouse, mousepad |
| `mous` | mouse, mousepad | mouse, mousepad |
| `mouse` | mouse, mousepad | mouse, mousepad |

Quan sát: mỗi khi gõ thêm một ký tự, tập khớp **chỉ co lại** (là tập con của tập trước) và vẫn là một **đoạn liên tiếp** trong danh sách đã sắp xếp.

In [7]:
sp = sorted(EX1[0])
for i in range(1, len("mouse") + 1):
    p = "mouse"[:i]
    match = [s for s in sp if s.startswith(p)]
    print(f"{p:6} khớp {match} -> {match[:3]}")

m      khớp ['mobile', 'moneypot', 'monitor', 'mouse', 'mousepad'] -> ['mobile', 'moneypot', 'monitor']
mo     khớp ['mobile', 'moneypot', 'monitor', 'mouse', 'mousepad'] -> ['mobile', 'moneypot', 'monitor']
mou    khớp ['mouse', 'mousepad'] -> ['mouse', 'mousepad']
mous   khớp ['mouse', 'mousepad'] -> ['mouse', 'mousepad']
mouse  khớp ['mouse', 'mousepad'] -> ['mouse', 'mousepad']


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** với mỗi tiền tố, duyệt **toàn bộ** `products`, lọc chuỗi khớp, sắp xếp, lấy 3 phần tử đầu.

In [8]:
def suggest_brute(products, searchWord):
    res = []
    for i in range(1, len(searchWord) + 1):
        p = searchWord[:i]
        match = sorted(s for s in products if s.startswith(p))
        res.append(match[:3])
    return res

for prods, w, want in examples:
    print(suggest_brute(prods, w) == want)

True
True
True
True


**Độ phức tạp:** gọi `n` = số sản phẩm, `L` = `len(searchWord)`, `S` = tổng độ dài sản phẩm. Mỗi tiền tố: lọc `O(n·L)` (so sánh tiền tố) + sắp xếp `O(k log k · L)` với `k` số khớp ⇒ tổng **`O(L·(n·L + n log n · L))`** worst case. Với `n = 1000`, `L = 1000` đã là rất nặng.

**Nút thắt:** (1) sắp xếp lặp lại `L` lần — chỉ cần sắp **một lần**; (2) lọc toàn bộ `n` chuỗi mỗi lần — tập khớp chỉ co lại, nên lọc trên tập trước đó, hoặc tìm thẳng đoạn khớp bằng nhị phân.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — sắp xếp một lần:** sau khi sắp, 3 gợi ý cho tiền tố `p` là **3 phần tử đầu tiên** của đoạn khớp `p`.

**Insight 2 — đoạn khớp liên tiếp & co lại:** gọi `[lo, hi)` là đoạn khớp `p`. Khi `p` dài thêm một ký tự, đoạn mới nằm **bên trong** đoạn cũ. Nên có thể:
- tìm `lo = bisect_left(sorted_products, p)` (Cách 3), hoặc
- giữ hai con trỏ `lo`, `hi` và chỉ dịch vào trong (Cách 4).

**Insight 3 — Trie:** mỗi nút Trie ứng với một tiền tố; nếu chèn sản phẩm **theo thứ tự đã sắp** và mỗi nút giữ tối đa 3 sản phẩm đầu tiên đi qua nó, thì đó đúng là 3 gợi ý cần tìm.

**Quy tắc "nếu… thì…":**
- Nếu tiền tố hiện tại không khớp sản phẩm nào ⇒ mọi tiền tố dài hơn cũng không khớp ⇒ thêm `[]` cho các bước còn lại.
- Nếu đoạn khớp có > 3 phần tử ⇒ chỉ lấy 3 phần tử đầu (đã nhỏ nhất theo từ điển).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp một lần + lọc dần

Giữ danh sách `cand` (đã sắp). Mỗi ký tự mới ở vị trí `i`: `cand = [s for s in cand if len(s) > i and s[i] == c]`. Chỉ so **một ký tự** mỗi lần vì phần trước đã khớp.

In [9]:
def suggest_filter(products, searchWord):
    cand = sorted(products)
    res = []
    for i, c in enumerate(searchWord):
        cand = [s for s in cand if len(s) > i and s[i] == c]
        res.append(cand[:3])
    return res

for prods, w, want in examples:
    print(suggest_filter(prods, w) == want)

True
True
True
True


**Độ phức tạp Cách 2:** sắp xếp `O(n log n · m)` (`m` = độ dài so sánh trung bình); lọc: tổng số lần kiểm tra ≤ `n·L` (mỗi bước duyệt các ứng viên còn lại) ⇒ `O(n log n · m + n·L)`. Bộ nhớ `O(n)`.

### 7.2 Cách 3 — Sắp xếp + `bisect` (tìm kiếm nhị phân)

Với mỗi tiền tố `p`: `i = bisect_left(sp, p)`; xét tối đa 3 phần tử `sp[i], sp[i+1], sp[i+2]`, giữ những phần tử `startswith(p)`. Vì đoạn khớp liên tiếp, khi gặp phần tử đầu tiên không khớp là có thể dừng.

Tối ưu nhỏ: đoạn khớp của `p` dài hơn nằm bên phải đoạn cũ ⇒ truyền `lo=i` cũ vào `bisect_left` để thu hẹp vùng tìm.

In [10]:
def suggest_bisect(products, searchWord):
    sp = sorted(products)
    res = []
    lo = 0
    p = ""
    for c in searchWord:
        p += c
        lo = bisect.bisect_left(sp, p, lo)
        res.append([s for s in sp[lo:lo + 3] if s.startswith(p)])
    return res

for prods, w, want in examples:
    print(suggest_bisect(prods, w) == want)

True
True
True
True


Bản nộp LeetCode của Cách 3:

In [11]:
class SolutionBisect:
    def suggestedProducts(self, products: list[str], searchWord: str) -> list[list[str]]:
        products = sorted(products)
        res, lo, prefix = [], 0, ""
        for c in searchWord:
            prefix += c
            lo = bisect.bisect_left(products, prefix, lo)
            res.append([s for s in products[lo:lo + 3] if s.startswith(prefix)])
        return res

print([SolutionBisect().suggestedProducts(p, w) == want for p, w, want in examples])

[True, True, True, True]


**Độ phức tạp Cách 3:** sắp xếp `O(n log n · m)`; mỗi ký tự: `bisect` `O(log n)` lần so sánh, mỗi lần so sánh chuỗi tốn tới `O(L)` ⇒ `O(L · log n · L)`; cộng xây `prefix` (`O(L²)` tổng) ⇒ **`O(n log n · m + L² log n)`**. Bộ nhớ `O(n)` cho bản sắp xếp (không sửa input) + `O(L)` mỗi danh sách kết quả.

### 7.3 Cách 4 — Sắp xếp + hai con trỏ

Giữ `lo = 0`, `hi = n − 1` trên danh sách đã sắp. Ở ký tự thứ `i` là `c`:
- dịch `lo` sang phải khi `sp[lo]` quá ngắn hoặc `sp[lo][i] != c`;
- dịch `hi` sang trái khi `sp[hi]` quá ngắn hoặc `sp[hi][i] != c`;
- gợi ý = `sp[lo : min(lo + 3, hi + 1)]`.

Mỗi con trỏ chỉ di chuyển một chiều ⇒ tổng số bước dịch `O(n)`.

In [12]:
def suggest_two_pointers(products, searchWord):
    sp = sorted(products)
    lo, hi = 0, len(sp) - 1
    res = []
    for i, c in enumerate(searchWord):
        while lo <= hi and (len(sp[lo]) <= i or sp[lo][i] != c):
            lo += 1
        while lo <= hi and (len(sp[hi]) <= i or sp[hi][i] != c):
            hi -= 1
        res.append(sp[lo:min(lo + 3, hi + 1)])
    return res

for prods, w, want in examples:
    print(suggest_two_pointers(prods, w) == want)

True
True
True
True


Vì sao chỉ cần so ký tự thứ `i`? Bất biến: mọi chuỗi trong `sp[lo..hi]` khớp `searchWord[:i]`. Trong đoạn này, các chuỗi có ký tự thứ `i` bằng `c` tạo thành một đoạn con liên tiếp (vì đã sắp xếp và có cùng tiền tố độ dài `i`) ⇒ chỉ cần cắt hai đầu.

**Độ phức tạp Cách 4:** sắp xếp `O(n log n · m)` + `O(n + L)` cho hai con trỏ ⇒ **`O(n log n · m + L)`** (không tính cắt danh sách kết quả, mỗi lần ≤ 3 phần tử). Bộ nhớ `O(n)`.

### 7.4 Cách 5 — Trie lưu sẵn top-3

1. Sắp xếp `products`.
2. Chèn lần lượt vào Trie; tại **mỗi nút** đi qua, nếu danh sách `top` của nút có < 3 phần tử thì thêm sản phẩm vào. Vì chèn theo thứ tự tăng dần, `top` luôn là 3 sản phẩm nhỏ nhất có tiền tố đó.
3. Duyệt `searchWord` trên Trie; khi rơi khỏi Trie thì mọi bước còn lại là `[]`.

In [13]:
def suggest_trie(products, searchWord):
    root = {}
    for s in sorted(products):
        node = root
        for ch in s:
            node = node.setdefault(ch, {})
            top = node.setdefault("#", [])
            if len(top) < 3:
                top.append(s)
    res = []
    node = root
    for ch in searchWord:
        node = node.get(ch) if node is not None else None
        res.append(node["#"] if node is not None else [])
    return res

for prods, w, want in examples:
    print(suggest_trie(prods, w) == want)

True
True
True
True


Bản nộp LeetCode của Cách 5 (Trie dạng lớp, rõ ràng hơn):

In [14]:
class TrieNode:
    __slots__ = ("children", "top")
    def __init__(self):
        self.children = {}
        self.top = []

class SolutionTrie:
    def suggestedProducts(self, products: list[str], searchWord: str) -> list[list[str]]:
        root = TrieNode()
        for s in sorted(products):
            node = root
            for ch in s:
                if ch not in node.children:
                    node.children[ch] = TrieNode()
                node = node.children[ch]
                if len(node.top) < 3:
                    node.top.append(s)
        res, node = [], root
        for ch in searchWord:
            node = node.children.get(ch) if node else None
            res.append(list(node.top) if node else [])
        return res

print([SolutionTrie().suggestedProducts(p, w) == want for p, w, want in examples])

[True, True, True, True]


**Độ phức tạp Cách 5:** sắp xếp `O(n log n · m)` + xây Trie `O(S)` (mỗi ký tự của mỗi sản phẩm một nút) + truy vấn `O(L)` ⇒ **`O(n log n · m + S + L)`**. Bộ nhớ `O(S)` nút (mỗi nút giữ ≤ 3 tham chiếu). Đây là cách tốt nhất nếu có **nhiều** `searchWord` cho cùng một `products`.

## 8. TRACE từng bước

Trace **Cách 4 (hai con trỏ)** trên Ví dụ 1, `sp = [mobile, moneypot, monitor, mouse, mousepad]` (chỉ số 0..4):

| `i` | ký tự | `lo` | `hi` | gợi ý |
|---|---|---|---|---|
| 0 | m | 0 | 4 | mobile, moneypot, monitor |
| 1 | o | 0 | 4 | mobile, moneypot, monitor |
| 2 | u | 3 | 4 | mouse, mousepad |
| 3 | s | 3 | 4 | mouse, mousepad |
| 4 | e | 3 | 4 | mouse, mousepad |

In [15]:
def trace_two_pointers(products, searchWord):
    sp = sorted(products)
    print("sp =", list(enumerate(sp)))
    lo, hi = 0, len(sp) - 1
    for i, c in enumerate(searchWord):
        moved = []
        while lo <= hi and (len(sp[lo]) <= i or sp[lo][i] != c):
            moved.append(f"lo bỏ {sp[lo]!r}"); lo += 1
        while lo <= hi and (len(sp[hi]) <= i or sp[hi][i] != c):
            moved.append(f"hi bỏ {sp[hi]!r}"); hi -= 1
        print(f"i={i} c={c!r}: {'; '.join(moved) or 'không dịch'} -> lo={lo}, hi={hi}, gợi ý={sp[lo:min(lo + 3, hi + 1)]}")

trace_two_pointers(*EX1[:2])

sp = [(0, 'mobile'), (1, 'moneypot'), (2, 'monitor'), (3, 'mouse'), (4, 'mousepad')]
i=0 c='m': không dịch -> lo=0, hi=4, gợi ý=['mobile', 'moneypot', 'monitor']
i=1 c='o': không dịch -> lo=0, hi=4, gợi ý=['mobile', 'moneypot', 'monitor']
i=2 c='u': lo bỏ 'mobile'; lo bỏ 'moneypot'; lo bỏ 'monitor' -> lo=3, hi=4, gợi ý=['mouse', 'mousepad']
i=3 c='s': không dịch -> lo=3, hi=4, gợi ý=['mouse', 'mousepad']
i=4 c='e': không dịch -> lo=3, hi=4, gợi ý=['mouse', 'mousepad']


In [16]:
def trace_bisect(products, searchWord):
    sp = sorted(products)
    lo, p = 0, ""
    for c in searchWord:
        p += c
        lo = bisect.bisect_left(sp, p, lo)
        window = sp[lo:lo + 3]
        print(f"p={p!r:9} bisect_left={lo}, cửa sổ={window}, giữ={[s for s in window if s.startswith(p)]}")

trace_bisect(*EX3[:2])

p='b'       bisect_left=0, cửa sổ=['baggage', 'bags', 'banner'], giữ=['baggage', 'bags', 'banner']
p='ba'      bisect_left=0, cửa sổ=['baggage', 'bags', 'banner'], giữ=['baggage', 'bags', 'banner']
p='bag'     bisect_left=0, cửa sổ=['baggage', 'bags', 'banner'], giữ=['baggage', 'bags']
p='bags'    bisect_left=1, cửa sổ=['bags', 'banner', 'box'], giữ=['bags']


## 9. Code nộp LeetCode

Chọn Cách 4 (sort + hai con trỏ): ngắn, `O(L)` sau khi sắp xếp, không sửa input.

In [17]:
class Solution:
    def suggestedProducts(self, products: list[str], searchWord: str) -> list[list[str]]:
        sp = sorted(products)
        lo, hi = 0, len(sp) - 1
        res = []
        for i, c in enumerate(searchWord):
            while lo <= hi and (len(sp[lo]) <= i or sp[lo][i] != c):
                lo += 1
            while lo <= hi and (len(sp[hi]) <= i or sp[hi][i] != c):
                hi -= 1
            res.append(sp[lo:min(lo + 3, hi + 1)])
        return res

for prods, w, want in examples:
    got = Solution().suggestedProducts(prods, w)
    print("OK" if got == want else "SAI", got)

OK [['mobile', 'moneypot', 'monitor'], ['mobile', 'moneypot', 'monitor'], ['mouse', 'mousepad'], ['mouse', 'mousepad'], ['mouse', 'mousepad']]
OK [['havana'], ['havana'], ['havana'], ['havana'], ['havana'], ['havana']]
OK [['baggage', 'bags', 'banner'], ['baggage', 'bags', 'banner'], ['baggage', 'bags'], ['bags']]
OK [[], [], [], [], [], [], []]


## 10. Vì sao thuật toán đúng

**Bổ đề (đoạn liên tiếp).** Trong danh sách `sp` đã sắp xếp, tập các chuỗi có tiền tố `p` là một đoạn liên tiếp, và đoạn này bắt đầu tại `bisect_left(sp, p)`.
Chứng minh: nếu `a ≤ b ≤ c` (theo từ điển) và `a`, `c` cùng bắt đầu bằng `p` thì `b` cũng bắt đầu bằng `p` (nếu `b` khác `p` ở vị trí đầu tiên `j < |p|` thì `b[j] < p[j]` ⇒ `b < a`, hoặc `b[j] > p[j]` ⇒ `b > c`; nếu `b` ngắn hơn `p` và là tiền tố của `p` thì `b < a`). Mọi chuỗi có tiền tố `p` đều `≥ p`. Xét chuỗi `x ≥ p` không bắt đầu bằng `p`: tại vị trí khác nhau đầu tiên `j < |p|` có `x[j] > p[j]`, nên `x` lớn hơn **mọi** chuỗi bắt đầu bằng `p`. Vậy giữa vị trí `bisect_left(sp, p)` và đoạn khớp không có chuỗi nào chen vào ⇒ đoạn khớp bắt đầu đúng tại đó.

**Đúng của 3 gợi ý.** 3 phần tử đầu của đoạn khớp là 3 chuỗi khớp nhỏ nhất theo từ điển — đúng yêu cầu.

**Cách 4.** Bất biến: sau bước `i`, `sp[lo..hi]` đúng bằng đoạn khớp `searchWord[:i+1]`. Bước `i` chỉ cần loại khỏi hai đầu những chuỗi có ký tự thứ `i` khác `c` (hoặc quá ngắn); theo bổ đề áp dụng cho tiền tố mới, phần còn lại liên tiếp nên không có chuỗi "xấu" nằm giữa.

**Cách 5.** Mỗi nút Trie ứng với một tiền tố; vì chèn theo thứ tự tăng dần, ba sản phẩm đầu tiên đi qua nút là ba sản phẩm nhỏ nhất có tiền tố đó.

## 11. Phân tích Time / Space

Ký hiệu: `n` = số sản phẩm (≤ 1000), `S` = tổng độ dài (≤ 2·10⁴), `m` = độ dài so sánh trung bình khi sắp xếp, `L` = `len(searchWord)` (≤ 1000).

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute force | `O(L·(n·L + k log k·L))` | `O(n)` |
| 2. Sort + lọc dần | `O(n log n·m + n·L)` | `O(n)` |
| 3. Sort + bisect | `O(n log n·m + L² log n)` | `O(n)` |
| 4. Sort + hai con trỏ | `O(n log n·m + n + L)` | `O(n)` |
| 5. Trie top-3 | `O(n log n·m + S + L)` | `O(S)` |

Phần chi phối thường là **sắp xếp**. Nếu có `Q` từ khoá truy vấn trên cùng `products`: Trie (`O(S)` một lần + `O(L)` mỗi truy vấn) hoặc bisect là lựa chọn tốt; Cách 2 tốn `O(n·L)` mỗi truy vấn.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = {
    "brute": suggest_brute,
    "filter": suggest_filter,
    "bisect": suggest_bisect,
    "bisect_submit": lambda p, w: SolutionBisect().suggestedProducts(p, w),
    "two_pointers": suggest_two_pointers,
    "trie": suggest_trie,
    "trie_submit": lambda p, w: SolutionTrie().suggestedProducts(p, w),
    "submit": lambda p, w: Solution().suggestedProducts(p, w),
}
edge = [
    (["a"], "a", [["a"]]),
    (["a", "b", "c"], "d", [[]]),
    (["ab", "abc", "abcd", "abcde"], "abcdef", [["ab", "abc", "abcd"], ["ab", "abc", "abcd"], ["abc", "abcd", "abcde"],
                                                 ["abcd", "abcde"], ["abcde"], []]),
    (["code", "codephone", "coddle", "coddles", "codes"], "coddle",
     [["coddle", "coddles", "code"], ["coddle", "coddles", "code"], ["coddle", "coddles", "code"],
      ["coddle", "coddles"], ["coddle", "coddles"], ["coddle", "coddles"]]),
    (["bb", "ba"], "bc", [["ba", "bb"], []]),            # hết khớp ở giữa
]
for prods, w, want in edge:
    assert suggest_brute(prods, w) == want, (prods, w)
for name, f in ALL.items():
    for prods, w, want in examples + edge:
        snapshot = prods[:]
        assert f(prods, w) == want, (name, prods, w, f(prods, w))
        assert prods == snapshot, (name, "đã sửa input")
print("Mọi cách đúng với", len(examples + edge), "test, không sửa input")

Mọi cách đúng với 9 test, không sửa input


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Dùng bảng chữ cái nhỏ (`abc`) để có nhiều tiền tố chung; sản phẩm **khác nhau** (đúng ràng buộc).

In [19]:
def random_case(rng):
    alpha = rng.choice(["ab", "abc", "abcd"])
    prods = set()
    for _ in range(rng.randint(1, 25)):
        prods.add("".join(rng.choice(alpha) for _ in range(rng.randint(1, 6))))
    prods = list(prods)
    rng.shuffle(prods)
    if rng.random() < 0.6:
        base = rng.choice(prods)
        w = base[:rng.randint(1, len(base))] + "".join(rng.choice(alpha) for _ in range(rng.randint(0, 3)))
    else:
        w = "".join(rng.choice(alpha) for _ in range(rng.randint(1, 7)))
    return prods, w

rng = random.Random(1268)
for _ in range(800):
    prods, w = random_case(rng)
    want = suggest_brute(prods, w)
    for name, f in ALL.items():
        assert f(prods, w) == want, (name, prods, w)
print("800 test ngẫu nhiên: mọi cách khớp brute force")

800 test ngẫu nhiên: mọi cách khớp brute force


### 12.3 Kích thước lớn

1000 sản phẩm có chung tiền tố dài (tổng độ dài ≈ 2·10⁴), `searchWord` dài 1000.

In [20]:
prefix = "a" * 8
prods = sorted({prefix + "".join(rng.choice("abcd") for _ in range(10)) for _ in range(1100)})[:1000]
rng.shuffle(prods)
w = prefix + "ab" + "c" * 990
print("n =", len(prods), "| tổng độ dài =", sum(map(len, prods)), "| len(searchWord) =", len(w))
ref = None
for name, f in ALL.items():
    t0 = time.perf_counter()
    res = f(prods, w)
    dt = time.perf_counter() - t0
    ref = res if ref is None else ref
    assert res == ref
    print(f"{name:14}: {dt:.3f}s")

n = 1000 | tổng độ dài = 18000 | len(searchWord) = 1000


brute         : 0.038s
filter        : 0.001s
bisect        : 0.000s
bisect_submit : 0.000s
two_pointers  : 0.000s
trie          : 0.003s


trie_submit   : 0.004s
submit        : 0.000s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên sắp xếp

Lấy 3 sản phẩm khớp **đầu tiên theo thứ tự input** thay vì nhỏ nhất theo từ điển.

In [21]:
def suggest_wrong1(products, searchWord):
    res = []
    for i in range(1, len(searchWord) + 1):
        p = searchWord[:i]
        res.append([s for s in products if s.startswith(p)][:3])     # SAI: không sort
    return res

print("Sai :", suggest_wrong1(*EX1[:2])[:2])
print("Đúng:", Solution().suggestedProducts(*EX1[:2])[:2])

Sai : [['mobile', 'mouse', 'moneypot'], ['mobile', 'mouse', 'moneypot']]
Đúng: [['mobile', 'moneypot', 'monitor'], ['mobile', 'moneypot', 'monitor']]


### Lỗi 2 — Dừng vòng lặp khi hết gợi ý (thiếu danh sách rỗng)

Kết quả phải có **đúng** `len(searchWord)` phần tử. Viết `break` khi Trie không còn nhánh ⇒ thiếu các `[]` ở cuối.

In [22]:
def suggest_wrong2(products, searchWord):
    root = {}
    for s in sorted(products):
        node = root
        for ch in s:
            node = node.setdefault(ch, {})
            node.setdefault("#", [])
            if len(node["#"]) < 3:
                node["#"].append(s)
    res, node = [], root
    for ch in searchWord:
        if ch not in node:
            break                                   # SAI: bỏ luôn các bước còn lại
        node = node[ch]
        res.append(node["#"])
    return res

got = suggest_wrong2(*EX1[:1], "mouz")
print("Sai :", got, "| độ dài", len(got))
print("Đúng:", Solution().suggestedProducts(EX1[0], "mouz"))

Sai : [['mobile', 'moneypot', 'monitor'], ['mobile', 'moneypot', 'monitor'], ['mouse', 'mousepad']] | độ dài 3
Đúng: [['mobile', 'moneypot', 'monitor'], ['mobile', 'moneypot', 'monitor'], ['mouse', 'mousepad'], []]


### Lỗi 3 — `bisect` rồi lấy luôn 3 phần tử, không kiểm tra tiền tố

`bisect_left` chỉ cho vị trí chèn; các phần tử từ đó trở đi có thể **không** bắt đầu bằng tiền tố.

In [23]:
def suggest_wrong3(products, searchWord):
    sp = sorted(products)
    res = []
    for i in range(1, len(searchWord) + 1):
        lo = bisect.bisect_left(sp, searchWord[:i])
        res.append(sp[lo:lo + 3])                     # SAI: thiếu startswith
    return res

print("Sai :", suggest_wrong3(*EX3[:2]))
print("Đúng:", Solution().suggestedProducts(*EX3[:2]))

Sai : [['baggage', 'bags', 'banner'], ['baggage', 'bags', 'banner'], ['baggage', 'bags', 'banner'], ['bags', 'banner', 'box']]
Đúng: [['baggage', 'bags', 'banner'], ['baggage', 'bags', 'banner'], ['baggage', 'bags'], ['bags']]


### Lỗi 4 — Hai con trỏ quên kiểm tra độ dài chuỗi

Viết `sp[lo][i] != c` mà không xét `len(sp[lo]) <= i` ⇒ `IndexError` khi một sản phẩm ngắn hơn phần đã gõ. Một "sửa vội" phổ biến là bọc `try/except` và **bỏ qua** chuỗi đó — nhưng lại bỏ qua sai chỗ (không dịch con trỏ), dẫn tới kết quả sai. Ta minh hoạ bản "sửa vội":

In [24]:
def suggest_wrong4(products, searchWord):
    sp = sorted(products)
    lo, hi = 0, len(sp) - 1
    res = []
    for i, c in enumerate(searchWord):
        try:
            while lo <= hi and sp[lo][i] != c:
                lo += 1
            while lo <= hi and sp[hi][i] != c:
                hi -= 1
        except IndexError:
            pass                                      # SAI: chuỗi ngắn vẫn nằm trong đoạn
        res.append(sp[lo:min(lo + 3, hi + 1)])
    return res

prods, w = ["ab", "abc", "abd"], "abd"
print("Sai :", suggest_wrong4(prods, w))
print("Đúng:", Solution().suggestedProducts(prods, w))

Sai : [['ab', 'abc', 'abd'], ['ab', 'abc', 'abd'], ['ab', 'abc', 'abd']]
Đúng: [['ab', 'abc', 'abd'], ['ab', 'abc', 'abd'], ['abd']]


### Lỗi 5 — Sắp xếp tại chỗ `products.sort()` (sửa input)

Không sai về kết quả trên LeetCode, nhưng làm thay đổi danh sách của người gọi — một "tác dụng phụ" dễ gây bug khi hàm được dùng lại.

In [25]:
def suggest_wrong5(products, searchWord):
    products.sort()                                  # tác dụng phụ: sửa input
    return suggest_filter(products, searchWord)

data = ["mouse", "mobile", "monitor"]
suggest_wrong5(data, "mo")
print("Danh sách của người gọi sau khi gọi hàm:", data, "(đã bị đổi thứ tự)")

Danh sách của người gọi sau khi gọi hàm: ['mobile', 'monitor', 'mouse'] (đã bị đổi thứ tự)


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"gợi ý theo tiền tố", "autocomplete", "top-k nhỏ nhất theo từ điển"** ⇒ sort + bisect / hai con trỏ, hoặc Trie.
- Nhớ: **sau khi sắp xếp, cùng tiền tố ⇒ đoạn liên tiếp** — mẹo này giải được rất nhiều bài chuỗi.
- `bisect_left(a, prefix)` cho điểm bắt đầu; kết thúc đoạn có thể tìm bằng `bisect_left(a, prefix + '{')` (`'{'` đứng ngay sau `'z'` trong bảng mã).
- Trie: lưu thông tin "tổng hợp" (top-k, số từ, …) ngay tại nút để trả lời `O(độ dài tiền tố)`.
- Luôn trả về **đủ** `len(searchWord)` danh sách, kể cả rỗng.

## 15. Biến thể của bài

1. **Top-k thay vì 3**: thay hằng số 3 bằng `k`.
2. **Xếp theo độ phổ biến** (bài 642 — Design Search Autocomplete System): mỗi sản phẩm có tần suất; Trie lưu top-k theo (−tần suất, chuỗi) hoặc dùng heap.
3. **Đếm số sản phẩm** có tiền tố cho mỗi bước: `bisect_left(a, p + '{') − bisect_left(a, p)`.
4. **Nhiều truy vấn** trên cùng `products` ⇒ dựng Trie/sort một lần (thiết kế lớp).
5. **Bài 208 — Implement Trie**, **1032 — Stream of Characters**: các bài Trie nền tảng.

Minh hoạ biến thể 2 và 3:

In [26]:
import heapq

def suggest_by_popularity(freq, searchWord, k=3):
    """freq: dict sản phẩm -> số lần được tìm. Ưu tiên tần suất cao, hoà thì theo từ điển."""
    res = []
    for i in range(1, len(searchWord) + 1):
        p = searchWord[:i]
        res.append(heapq.nsmallest(k, (s for s in freq if s.startswith(p)), key=lambda s: (-freq[s], s)))
    return res

def count_per_prefix(products, searchWord):
    sp = sorted(products)
    out = []
    for i in range(1, len(searchWord) + 1):
        p = searchWord[:i]
        out.append(bisect.bisect_left(sp, p + "{") - bisect.bisect_left(sp, p))
    return out

print(suggest_by_popularity({"mobile": 1, "mouse": 5, "moneypot": 2, "monitor": 5, "mousepad": 3}, "mou"))
print(count_per_prefix(*EX1[:2]))

[['monitor', 'mouse', 'mousepad'], ['monitor', 'mouse', 'mousepad'], ['mouse', 'mousepad']]
[5, 5, 2, 2, 2]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Mỗi tiền tố: lọc + sort toàn bộ | `O(L·n·L log n)` | `O(n)` | Kiểm chứng |
| 2. Sort + lọc dần | Sort 1 lần, lọc tập ứng viên co dần | `O(n log n·m + n·L)` | `O(n)` | Dễ viết, đủ nhanh |
| 3. Sort + bisect | Tìm đầu đoạn khớp bằng nhị phân | `O(n log n·m + L² log n)` | `O(n)` | Nhiều truy vấn, không muốn dựng Trie |
| 4. Sort + hai con trỏ | Co `[lo, hi]` theo từng ký tự | `O(n log n·m + n + L)` | `O(n)` | Mặc định cho 1 truy vấn |
| 5. Trie top-3 | Mỗi nút lưu sẵn 3 gợi ý | `O(n log n·m + S + L)` | `O(S)` | Hệ thống autocomplete thực sự |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().suggestedProducts(["apple", "app", "apricot", "banana"], "apr")` = ?

### Bài 2 (tính tay)
Với `products = ["car", "card", "care", "cart", "cat"]`, `searchWord = "cars"`, chạy tay hai con trỏ: ghi `lo`, `hi` và gợi ý sau mỗi ký tự.

### Bài 3 (điền TODO)
Hoàn thiện `suggest_topk(products, searchWord, k)` — tổng quát hoá Cách 3 cho `k` gợi ý.

### Bài 4 (tự viết lại)
Viết lại Cách 5 sao cho mỗi nút Trie **không** lưu top-3 mà chỉ lưu các con; khi truy vấn, DFS theo thứ tự chữ cái để lấy 3 từ đầu tiên: `suggest_trie_dfs(products, searchWord)`.

### Bài 5 (mini challenge)
Thiết kế lớp `AutoComplete(products)` với phương thức `type(ch)` trả về gợi ý sau khi gõ thêm ký tự `ch`, và `reset()` để bắt đầu từ khoá mới. Mỗi `type` phải chạy `O(1)` (không tính độ dài kết quả).

In [27]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def suggest_topk(products, searchWord, k):
    # TODO: sort, bisect_left với lo tăng dần, lấy tối đa k phần tử có startswith(prefix)
    pass

# Bài 4 — TODO
def suggest_trie_dfs(products, searchWord):
    # TODO 1: dựng Trie chỉ có children + cờ kết thúc từ
    # TODO 2: với mỗi tiền tố, DFS theo thứ tự chữ cái, dừng khi đủ 3 từ
    pass

# Bài 5 — TODO
class AutoComplete:
    def __init__(self, products):
        # TODO: dựng Trie top-3 (như Cách 5) và con trỏ nút hiện tại
        pass
    def type(self, ch):
        # TODO
        pass
    def reset(self):
        # TODO
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Sắp xếp: `app, apple, apricot, banana`. (2) `a`, `ap`: ba từ đầu. (3) `apr`: chỉ `apricot` ⇒ `[["app","apple","apricot"],["app","apple","apricot"],["apricot"]]`.

**Bài 2.** (1) Đã sắp xếp sẵn: `car, card, care, cart, cat` (0..4). (2) `c`, `a`: `lo=0, hi=4`; `r`: `hi` bỏ `cat` ⇒ `hi=3`. (3) `s`: `lo` bỏ `car` (quá ngắn), `card`, `care`, `cart` ⇒ đoạn rỗng ⇒ `[]`.

**Bài 3.** (1) Thay `3` bằng `k`. (2) Giữ `lo` tăng dần. (3) Kiểm tra `k = 3` trùng `Solution`.

**Bài 4.** (1) Nút: `{'children': {}, 'end': False}` hoặc dict với khoá `'$'`. (2) DFS duyệt `sorted(children)`, ghi từ khi gặp cờ kết thúc (từ ngắn hơn đứng trước — kiểm tra cờ **trước** khi đi xuống con). (3) Dừng khi đủ 3.

**Bài 5.** (1) Lưu `self.node` (nút hiện tại, `None` nếu đã rơi khỏi Trie). (2) `type`: đi xuống một con, trả `top` của nút hoặc `[]`. (3) `reset`: `self.node = self.root`.

## 19. Đáp án tham khảo

In [28]:
# Bài 1
assert Solution().suggestedProducts(["apple", "app", "apricot", "banana"], "apr") == \
    [["app", "apple", "apricot"], ["app", "apple", "apricot"], ["apricot"]]
# Bài 2
assert Solution().suggestedProducts(["car", "card", "care", "cart", "cat"], "cars") == \
    [["car", "card", "care"], ["car", "card", "care"], ["car", "card", "care"], []]

# Bài 3
def suggest_topk_sol(products, searchWord, k):
    sp = sorted(products)
    res, lo, p = [], 0, ""
    for c in searchWord:
        p += c
        lo = bisect.bisect_left(sp, p, lo)
        out = []
        j = lo
        while j < len(sp) and len(out) < k and sp[j].startswith(p):
            out.append(sp[j]); j += 1
        res.append(out)
    return res

# Bài 4
def suggest_trie_dfs_sol(products, searchWord):
    root = {}
    for s in products:
        node = root
        for ch in s:
            node = node.setdefault(ch, {})
        node["$"] = True
    def collect(node, path, out):
        if len(out) == 3:
            return
        if "$" in node:
            out.append(path)
        for ch in sorted(k for k in node if k != "$"):
            if len(out) == 3:
                return
            collect(node[ch], path + ch, out)
    res, node, p = [], root, ""
    for ch in searchWord:
        p += ch
        node = node.get(ch) if node is not None else None
        out = []
        if node is not None:
            collect(node, p, out)
        res.append(out)
    return res

# Bài 5
class AutoCompleteSol:
    def __init__(self, products):
        self.root = {}
        for s in sorted(products):
            node = self.root
            for ch in s:
                node = node.setdefault(ch, {})
                top = node.setdefault("#", [])
                if len(top) < 3:
                    top.append(s)
        self.node = self.root
    def type(self, ch):
        self.node = self.node.get(ch) if self.node is not None else None
        return list(self.node["#"]) if self.node is not None else []
    def reset(self):
        self.node = self.root

rng = random.Random(21)
for _ in range(300):
    prods, w = random_case(rng)
    want = Solution().suggestedProducts(prods, w)
    assert suggest_topk_sol(prods, w, 3) == want
    assert suggest_trie_dfs_sol(prods, w) == want
    k = rng.randint(1, 5)
    assert suggest_topk_sol(prods, w, k) == [sorted(s for s in prods if s.startswith(w[:i]))[:k] for i in range(1, len(w) + 1)]
    ac = AutoCompleteSol(prods)
    assert [ac.type(c) for c in w] == want
    ac.reset()
    assert [ac.type(c) for c in w[:2]] == want[:2]
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template sort + hai con trỏ:**

```python
sp = sorted(products)
lo, hi = 0, len(sp) - 1
res = []
for i, c in enumerate(searchWord):
    while lo <= hi and (len(sp[lo]) <= i or sp[lo][i] != c): lo += 1
    while lo <= hi and (len(sp[hi]) <= i or sp[hi][i] != c): hi -= 1
    res.append(sp[lo:min(lo + 3, hi + 1)])
```

**Template sort + bisect:** `lo = bisect_left(sp, prefix, lo)`; lấy ≤ 3 phần tử từ `lo` có `startswith(prefix)`.

**Template Trie top-k:** chèn theo thứ tự đã sắp; mỗi nút đi qua thêm từ vào `top` nếu `len(top) < k`.

**Checklist trước khi submit:**
- [ ] Sắp xếp `products` (bản sao, không sửa input).
- [ ] Kết quả có **đúng** `len(searchWord)` danh sách (kể cả `[]`).
- [ ] `bisect` phải kèm kiểm tra `startswith`.
- [ ] Hai con trỏ kiểm tra `len(s) <= i` trước khi đọc `s[i]`.
- [ ] Test: không khớp ngay từ ký tự đầu; khớp rồi mất khớp giữa chừng; sản phẩm là tiền tố của sản phẩm khác.
- [ ] Đúng chữ ký: `suggestedProducts(self, products, searchWord) -> list[list[str]]`.